# SOC Assistant - Advanced Model Training on Google Colab

This notebook trains ML models using your Mininet-generated network data with comprehensive reporting.

**Features:**
- Uploads and processes Mininet CSV data
- Trains Random Forest, XGBoost, and Ensemble models
- Generates comprehensive evaluation reports
- Creates visualizations and performance metrics
- Downloads trained models and reports

**Expected Data:** ~10,000 mixed records (normal + compromised traffic)

## Step 1: Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost imbalanced-learn matplotlib seaborn plotly

## Step 2: Upload Your Mininet Dataset

Upload your CSV file from:
`mininet_data_generation/data_capture/processed/synthetic_dataset_*.csv`

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
from datetime import datetime

print("Please upload your Mininet CSV dataset...")
uploaded = files.upload()

# Get the uploaded filename
csv_file = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {csv_file}")

# Load data
print("\nLoading dataset...")
df = pd.read_csv(csv_file)

print(f"✓ Loaded {len(df)} samples")
print(f"  Columns: {len(df.columns)}")
print(f"\nDataset Info:")
print(df.info())
print(f"\nFirst few rows:")
df.head()

## Step 3: Data Analysis & Preprocessing

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("DATA ANALYSIS")
print("="*60)

# Check for label column
if 'label' not in df.columns:
    print("\n⚠ Warning: 'label' column not found!")
    print("Available columns:", df.columns.tolist())
else:
    # Class distribution
    print("\nClass Distribution:")
    print(df['label'].value_counts())
    print(f"\nNormal: {len(df[df['label'] == 0])} ({len(df[df['label'] == 0])/len(df)*100:.1f}%)")
    print(f"Attack: {len(df[df['label'] == 1])} ({len(df[df['label'] == 1])/len(df)*100:.1f}%)")
    
    # Visualize class distribution
    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    df['label'].value_counts().plot(kind='bar', color=['green', 'red'])
    plt.title('Class Distribution')
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.xticks([0, 1], ['Normal', 'Attack'], rotation=0)
    
    # Attack type distribution
    if 'attack_type' in df.columns:
        plt.subplot(1, 2, 2)
        attack_counts = df['attack_type'].value_counts()
        attack_counts.plot(kind='bar', color='coral')
        plt.title('Attack Type Distribution')
        plt.xlabel('Attack Type')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        
        print("\nAttack Types:")
        for attack_type, count in attack_counts.items():
            print(f"  {attack_type}: {count}")
    
    plt.tight_layout()
    plt.savefig('class_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

# Missing values
print("\nMissing Values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("  None")

# Data types
print("\nData Types:")
print(df.dtypes.value_counts())

In [ ]:
print("\n" + "="*60)
print("PREPROCESSING")
print("="*60)

# Separate features and labels
print("\nSeparating features and labels...")
X = df.drop(['label', 'attack_type'], axis=1, errors='ignore')
y = df['label']

# Store attack types for analysis
attack_types = df['attack_type'] if 'attack_type' in df.columns else None

# Drop non-numeric columns
non_numeric_cols = X.select_dtypes(include=['object']).columns.tolist()
if non_numeric_cols:
    print(f"Dropping non-numeric columns: {non_numeric_cols}")
    X = X.drop(columns=non_numeric_cols)

# Handle missing values
X = X.fillna(0)

# Handle infinite values
X = X.replace([np.inf, -np.inf], 0)

# Ensure all columns are numeric
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

print(f"\n✓ Features: {len(X.columns)}")
print(f"✓ Samples: {len(X)}")
print(f"✓ Normal: {sum(y == 0)}, Attack: {sum(y == 1)}")

print("\nFeature List:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i}. {col}")

## Step 4: Train/Validation/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

print("\n" + "="*60)
print("DATA SPLITTING")
print("="*60)

# Split: 60% train, 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nTrain Set: {len(X_train)} samples")
print(f"  Normal: {sum(y_train == 0)}, Attack: {sum(y_train == 1)}")

print(f"\nValidation Set: {len(X_val)} samples")
print(f"  Normal: {sum(y_val == 0)}, Attack: {sum(y_val == 1)}")

print(f"\nTest Set: {len(X_test)} samples")
print(f"  Normal: {sum(y_test == 0)}, Attack: {sum(y_test == 1)}")

# Visualize split
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (y_split, title) in zip(axes, [(y_train, 'Train'), (y_val, 'Validation'), (y_test, 'Test')]):
    counts = y_split.value_counts()
    ax.bar(['Normal', 'Attack'], [counts.get(0, 0), counts.get(1, 0)], color=['green', 'red'])
    ax.set_title(f'{title} Set')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('data_split.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 5: Feature Scaling & Selection

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif

print("\n" + "="*60)
print("FEATURE ENGINEERING")
print("="*60)

# Scale features
print("\nScaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print("✓ Features scaled")

# Feature selection
print("\nSelecting top features...")
k_features = min(30, X_train.shape[1])
selector = SelectKBest(mutual_info_classif, k=k_features)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_val_selected = selector.transform(X_val_scaled)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X.columns[selector.get_support()].tolist()
print(f"✓ Selected {len(selected_features)} features")

# Feature importance scores
feature_scores = pd.DataFrame({
    'feature': X.columns,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print("\nTop 10 Features by Importance:")
for i, row in feature_scores.head(10).iterrows():
    print(f"  {row['feature']}: {row['score']:.4f}")

# Visualize feature importance
plt.figure(figsize=(12, 6))
top_features = feature_scores.head(20)
plt.barh(range(len(top_features)), top_features['score'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance Score')
plt.title('Top 20 Feature Importance Scores')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 6: Handle Class Imbalance with SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

print("\n" + "="*60)
print("CLASS BALANCING")
print("="*60)

print(f"\nBefore SMOTE:")
print(f"  Total: {len(X_train_selected)}")
print(f"  Normal: {sum(y_train == 0)}")
print(f"  Attack: {sum(y_train == 1)}")
print(f"  Ratio: {sum(y_train == 0) / sum(y_train == 1):.2f}:1")

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train)

print(f"\nAfter SMOTE:")
print(f"  Total: {len(X_train_balanced)}")
print(f"  Normal: {sum(y_train_balanced == 0)}")
print(f"  Attack: {sum(y_train_balanced == 1)}")
print(f"  Ratio: 1:1 (balanced)")

# Visualize balancing
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(['Normal', 'Attack'], [sum(y_train == 0), sum(y_train == 1)], color=['green', 'red'])
ax1.set_title('Before SMOTE')
ax1.set_ylabel('Count')

ax2.bar(['Normal', 'Attack'], [sum(y_train_balanced == 0), sum(y_train_balanced == 1)], color=['green', 'red'])
ax2.set_title('After SMOTE')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.savefig('smote_balancing.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 7: Train Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
import time

print("\n" + "="*60)
print("TRAINING RANDOM FOREST")
print("="*60)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\nTraining Random Forest...")
start_time = time.time()
rf_model.fit(X_train_balanced, y_train_balanced)
training_time = time.time() - start_time
print(f"✓ Training completed in {training_time:.2f} seconds")

# Cross-validation
print("\nPerforming 5-fold cross-validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X_train_balanced, y_train_balanced, cv=cv, scoring='f1', n_jobs=-1)
print(f"CV F1 Scores: {cv_scores}")
print(f"Mean CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Validation set performance
val_score = rf_model.score(X_val_selected, y_val)
print(f"\nValidation Accuracy: {val_score:.4f}")

## Step 8: Train XGBoost Model

In [ ]:
import xgboost as xgb

print("\n" + "="*60)
print("TRAINING XGBOOST")
print("="*60)

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

print("\nTraining XGBoost...")
start_time = time.time()
xgb_model.fit(X_train_balanced, y_train_balanced)
training_time = time.time() - start_time
print(f"✓ Training completed in {training_time:.2f} seconds")

# Cross-validation
print("\nPerforming 5-fold cross-validation...")
cv_scores = cross_val_score(xgb_model, X_train_balanced, y_train_balanced, cv=cv, scoring='f1', n_jobs=-1)
print(f"CV F1 Scores: {cv_scores}")
print(f"Mean CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Validation set performance
val_score = xgb_model.score(X_val_selected, y_val)
print(f"\nValidation Accuracy: {val_score:.4f}")

## Step 9: Create Ensemble Model

In [ ]:
from sklearn.ensemble import VotingClassifier

print("\n" + "="*60)
print("CREATING ENSEMBLE")
print("="*60)

ensemble_model = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('xgb', xgb_model)
    ],
    voting='soft',
    n_jobs=-1
)

print("\nTraining ensemble...")
start_time = time.time()
ensemble_model.fit(X_train_balanced, y_train_balanced)
training_time = time.time() - start_time
print(f"✓ Ensemble created in {training_time:.2f} seconds")

# Validation set performance
val_score = ensemble_model.score(X_val_selected, y_val)
print(f"\nValidation Accuracy: {val_score:.4f}")

## Step 10: Comprehensive Model Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

print("\n" + "="*60)
print("MODEL EVALUATION ON TEST SET")
print("="*60)

# Predictions
y_pred_rf = rf_model.predict(X_test_selected)
y_pred_xgb = xgb_model.predict(X_test_selected)
y_pred_ensemble = ensemble_model.predict(X_test_selected)

y_pred_proba_rf = rf_model.predict_proba(X_test_selected)[:, 1]
y_pred_proba_xgb = xgb_model.predict_proba(X_test_selected)[:, 1]
y_pred_proba_ensemble = ensemble_model.predict_proba(X_test_selected)[:, 1]

# Calculate metrics for all models
models = {
    'Random Forest': (y_pred_rf, y_pred_proba_rf),
    'XGBoost': (y_pred_xgb, y_pred_proba_xgb),
    'Ensemble': (y_pred_ensemble, y_pred_proba_ensemble)
}

results = {}
for name, (y_pred, y_pred_proba) in models.items():
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba)
    }

# Display results
results_df = pd.DataFrame(results).T
print("\nModel Performance Comparison:")
print(results_df.to_string())

# Detailed report for ensemble
print("\n" + "="*60)
print("ENSEMBLE MODEL - DETAILED REPORT")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_ensemble, target_names=['Normal', 'Attack'], digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_ensemble)
print("\nConfusion Matrix:")
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives: {cm[1,1]}")

## Step 11: Generate Comprehensive Visualizations

In [ ]:
# 1. Confusion Matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, (y_pred, _)) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'])
    ax.set_title(f'{name}\nAccuracy: {accuracy_score(y_test, y_pred):.4f}')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. ROC Curves
plt.figure(figsize=(10, 8))

for name, (_, y_pred_proba) in models.items():
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Precision-Recall Curves
plt.figure(figsize=(10, 8))

for name, (_, y_pred_proba) in models.items():
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    ap = average_precision_score(y_test, y_pred_proba)
    plt.plot(recall, precision, label=f'{name} (AP = {ap:.4f})', linewidth=2)

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower left', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('precision_recall_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# 4. Model Performance Comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC AUC']

for ax, metric, metric_name in zip(axes.flat, metrics, metric_names):
    values = [results[model][metric] for model in results.keys()]
    bars = ax.bar(results.keys(), values, color=['skyblue', 'lightcoral', 'lightgreen'])
    ax.set_title(metric_name, fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1.1])
    ax.set_ylabel('Score')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=10)

# Hide the last subplot
axes.flat[-1].axis('off')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ All visualizations generated!")

## Step 12: Attack Type Analysis

In [ ]:
if attack_types is not None:
    print("\n" + "="*60)
    print("ATTACK TYPE ANALYSIS")
    print("="*60)
    
    # Get attack types for test set
    test_indices = y_test.index
    test_attack_types = attack_types.loc[test_indices]
    
    # Performance by attack type
    attack_performance = {}
    for attack_type in test_attack_types.unique():
        mask = test_attack_types == attack_type
        if mask.sum() > 0:
            y_true_attack = y_test[mask]
            y_pred_attack = y_pred_ensemble[mask]
            
            attack_performance[attack_type] = {
                'count': mask.sum(),
                'accuracy': accuracy_score(y_true_attack, y_pred_attack),
                'precision': precision_score(y_true_attack, y_pred_attack, zero_division=0),
                'recall': recall_score(y_true_attack, y_pred_attack, zero_division=0),
                'f1': f1_score(y_true_attack, y_pred_attack, zero_division=0)
            }
    
    attack_perf_df = pd.DataFrame(attack_performance).T
    print("\nPerformance by Attack Type:")
    print(attack_perf_df.to_string())
    
    # Visualize
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    titles = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    
    for ax, metric, title in zip(axes.flat, metrics, titles):
        attack_perf_df[metric].plot(kind='bar', ax=ax, color='coral')
        ax.set_title(f'{title} by Attack Type', fontweight='bold')
        ax.set_ylabel('Score')
        ax.set_ylim([0, 1.1])
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('attack_type_performance.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("\n⚠ Attack type information not available")

## Step 13: Generate Comprehensive Report

In [ ]:
import json

print("\n" + "="*60)
print("GENERATING COMPREHENSIVE REPORT")
print("="*60)

# Create comprehensive report
report = {
    'training_date': datetime.now().isoformat(),
    'dataset': {
        'total_samples': len(df),
        'normal_samples': int(sum(df['label'] == 0)),
        'attack_samples': int(sum(df['label'] == 1)),
        'features': len(X.columns),
        'selected_features': len(selected_features),
        'feature_list': selected_features
    },
    'data_split': {
        'train': len(X_train),
        'validation': len(X_val),
        'test': len(X_test)
    },
    'models': {
        name: {
            'accuracy': float(metrics['accuracy']),
            'precision': float(metrics['precision']),
            'recall': float(metrics['recall']),
            'f1_score': float(metrics['f1']),
            'roc_auc': float(metrics['roc_auc'])
        }
        for name, metrics in results.items()
    },
    'confusion_matrix': {
        'true_negatives': int(cm[0,0]),
        'false_positives': int(cm[0,1]),
        'false_negatives': int(cm[1,0]),
        'true_positives': int(cm[1,1])
    }
}

if attack_types is not None:
    report['attack_types'] = {
        attack_type: {
            'count': int(perf['count']),
            'accuracy': float(perf['accuracy']),
            'precision': float(perf['precision']),
            'recall': float(perf['recall']),
            'f1_score': float(perf['f1'])
        }
        for attack_type, perf in attack_performance.items()
    }

# Save JSON report
with open('training_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("\n✓ Report saved to training_report.json")

# Generate HTML report
html_report = f"""
<!DOCTYPE html>
<html>
<head>
    <title>SOC Assistant - Training Report</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }}
        .container {{ background: white; padding: 30px; border-radius: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }}
        h1 {{ color: #2c3e50; border-bottom: 3px solid #3498db; padding-bottom: 10px; }}
        h2 {{ color: #34495e; margin-top: 30px; }}
        table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
        th, td {{ border: 1px solid #ddd; padding: 12px; text-align: left; }}
        th {{ background-color: #3498db; color: white; }}
        tr:nth-child(even) {{ background-color: #f2f2f2; }}
        .metric {{ display: inline-block; margin: 10px; padding: 15px; background: #ecf0f1; border-radius: 5px; }}
        .metric-value {{ font-size: 24px; font-weight: bold; color: #2980b9; }}
        .metric-label {{ font-size: 14px; color: #7f8c8d; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>SOC Assistant - Model Training Report</h1>
        <p><strong>Training Date:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        
        <h2>Dataset Summary</h2>
        <div class="metric">
            <div class="metric-value">{len(df)}</div>
            <div class="metric-label">Total Samples</div>
        </div>
        <div class="metric">
            <div class="metric-value">{sum(df['label'] == 0)}</div>
            <div class="metric-label">Normal Traffic</div>
        </div>
        <div class="metric">
            <div class="metric-value">{sum(df['label'] == 1)}</div>
            <div class="metric-label">Attack Traffic</div>
        </div>
        <div class="metric">
            <div class="metric-value">{len(selected_features)}</div>
            <div class="metric-label">Selected Features</div>
        </div>
        
        <h2>Model Performance</h2>
        <table>
            <tr>
                <th>Model</th>
                <th>Accuracy</th>
                <th>Precision</th>
                <th>Recall</th>
                <th>F1-Score</th>
                <th>ROC AUC</th>
            </tr>
            {''.join([f"""
            <tr>
                <td><strong>{name}</strong></td>
                <td>{metrics['accuracy']:.4f}</td>
                <td>{metrics['precision']:.4f}</td>
                <td>{metrics['recall']:.4f}</td>
                <td>{metrics['f1']:.4f}</td>
                <td>{metrics['roc_auc']:.4f}</td>
            </tr>
            """ for name, metrics in results.items()])}
        </table>
        
        <h2>Confusion Matrix (Ensemble Model)</h2>
        <table style="width: 50%;">
            <tr>
                <th></th>
                <th>Predicted Normal</th>
                <th>Predicted Attack</th>
            </tr>
            <tr>
                <th>Actual Normal</th>
                <td style="background: #d4edda;">{cm[0,0]}</td>
                <td style="background: #f8d7da;">{cm[0,1]}</td>
            </tr>
            <tr>
                <th>Actual Attack</th>
                <td style="background: #f8d7da;">{cm[1,0]}</td>
                <td style="background: #d4edda;">{cm[1,1]}</td>
            </tr>
        </table>
        
        <h2>Selected Features</h2>
        <ol>
            {''.join([f'<li>{feature}</li>' for feature in selected_features])}
        </ol>
    </div>
</body>
</html>
"""

with open('training_report.html', 'w') as f:
    f.write(html_report)

print("✓ HTML report saved to training_report.html")
print("\n✓ All reports generated!")

## Step 14: Save Models

In [ ]:
import joblib

print("\n" + "="*60)
print("SAVING MODELS")
print("="*60)

# Save models
joblib.dump(ensemble_model, 'mininet_ensemble_model.pkl')
joblib.dump(rf_model, 'mininet_random_forest_model.pkl')
joblib.dump(xgb_model, 'mininet_xgboost_model.pkl')
joblib.dump(scaler, 'mininet_scaler.pkl')
joblib.dump(selector, 'mininet_feature_selector.pkl')
joblib.dump(selected_features, 'mininet_feature_columns.pkl')

# Save metadata
metadata = {
    'accuracy': float(results['Ensemble']['accuracy']),
    'precision': float(results['Ensemble']['precision']),
    'recall': float(results['Ensemble']['recall']),
    'f1_score': float(results['Ensemble']['f1']),
    'roc_auc': float(results['Ensemble']['roc_auc']),
    'n_features': len(selected_features),
    'training_date': datetime.now().isoformat(),
    'n_samples': len(df),
    'n_normal': int(sum(df['label'] == 0)),
    'n_attack': int(sum(df['label'] == 1))
}
joblib.dump(metadata, 'mininet_model_metadata.pkl')

print("\n✓ Saved 7 model files:")
print("  1. mininet_ensemble_model.pkl")
print("  2. mininet_random_forest_model.pkl")
print("  3. mininet_xgboost_model.pkl")
print("  4. mininet_scaler.pkl")
print("  5. mininet_feature_selector.pkl")
print("  6. mininet_feature_columns.pkl")
print("  7. mininet_model_metadata.pkl")

print("\n✓ Generated 10+ visualization files")
print("✓ Generated JSON and HTML reports")

## Step 15: Download All Files

In [ ]:
from google.colab import files
import os

print("\n" + "="*60)
print("DOWNLOADING FILES")
print("="*60)

# List all files to download
files_to_download = [
    # Models
    'mininet_ensemble_model.pkl',
    'mininet_random_forest_model.pkl',
    'mininet_xgboost_model.pkl',
    'mininet_scaler.pkl',
    'mininet_feature_selector.pkl',
    'mininet_feature_columns.pkl',
    'mininet_model_metadata.pkl',
    # Reports
    'training_report.json',
    'training_report.html',
    # Visualizations
    'class_distribution.png',
    'data_split.png',
    'feature_importance.png',
    'smote_balancing.png',
    'confusion_matrices.png',
    'roc_curves.png',
    'precision_recall_curves.png',
    'model_comparison.png',
    'attack_type_performance.png'
]

print("\nDownloading files...")
for file in files_to_download:
    if os.path.exists(file):
        print(f"  Downloading {file}...")
        files.download(file)
    else:
        print(f"  ⚠ {file} not found")

print("\n" + "="*60)
print("✓ TRAINING COMPLETE!")
print("="*60)
print("\nNext steps:")
print("1. Upload model files to: /home/ongera/projects/SOC-assistant/models/")
print("2. Review reports and visualizations")
print("3. Restart dashboard: python src/dashboard/server.py")
print("4. Access: http://localhost:5000")
print("="*60)